In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import classification_report, roc_auc_score

In [3]:
df = pd.read_csv('Customer_Churn_Data_V3.csv')

In [4]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Contract_Month-to-month,Contract_One year,Contract_Two year,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [5]:
features = df.drop(columns = ['Churn'])
target = df['Churn']

In [6]:
features_train,features_test,target_train,target_test = train_test_split(features,target,test_size=0.2,random_state=42)

In [7]:
smote = SMOTE(random_state=42)
features_train,target_train = smote.fit_resample(features_train,target_train)

In [8]:
scaler = StandardScaler()
features_train = scaler.fit_transform(features_train)
features_test = scaler.transform(features_test)

In [9]:
features_train_tensor = torch.tensor(features_train, dtype=torch.float32)
features_test_tensor = torch.tensor(features_test, dtype=torch.float32)
target_train_tensor = torch.tensor(target_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)
target_test_tensor = torch.tensor(target_test.to_numpy(), dtype=torch.float32).reshape(-1, 1)

In [10]:
train_dataset = TensorDataset(features_train_tensor, target_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(features_test_tensor, target_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32)

In [11]:
class ChurnANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(40, 11)
        self.fc2 = nn.Linear(11, 6)
        self.fc3 = nn.Linear(6, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

In [12]:
model = ChurnANN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [13]:
epochs = 50

for epoch in range(epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}')

Epoch 10/50 | Loss: 0.2905
Epoch 20/50 | Loss: 0.2074
Epoch 30/50 | Loss: 0.0825
Epoch 40/50 | Loss: 0.1282
Epoch 50/50 | Loss: 0.1771


In [14]:
model.eval()
with torch.no_grad():
    y_pred = model(features_test_tensor)
    y_pred_labels = (y_pred > 0.5).float()

y_pred_np = y_pred_labels.numpy()
y_test_np = target_test_tensor.numpy()

print(accuracy_score(y_test_np, y_pred_np))
print(confusion_matrix(y_test_np, y_pred_np))

0.7647476901208244
[[844 189]
 [142 232]]


In [15]:
## Improvement  model
class ChurnANN2(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(40, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

In [16]:
model2 = ChurnANN2()
criterion = nn.BCELoss()
optimizer = optim.Adam(model2.parameters(), lr=0.0001)

In [17]:
epochs = 100

for epoch in range(epochs):
    model2.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        y_pred = model2(X_batch)
        loss = criterion(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Avg Loss: {epoch_loss/len(train_loader):.6f}')

Epoch 10/100 | Avg Loss: 0.445453
Epoch 20/100 | Avg Loss: 0.430061
Epoch 30/100 | Avg Loss: 0.418878
Epoch 40/100 | Avg Loss: 0.410678
Epoch 50/100 | Avg Loss: 0.399559
Epoch 60/100 | Avg Loss: 0.390402
Epoch 70/100 | Avg Loss: 0.381337
Epoch 80/100 | Avg Loss: 0.374286
Epoch 90/100 | Avg Loss: 0.366910
Epoch 100/100 | Avg Loss: 0.359528


In [18]:
model2.eval()
with torch.no_grad():
    y_pred = model2(features_test_tensor)
    y_pred_labels = (y_pred > 0.5).float()

y_pred_np = y_pred_labels.numpy()
y_test_np = target_test_tensor.numpy()

print(accuracy_score(y_test_np, y_pred_np))
print(confusion_matrix(y_test_np, y_pred_np))

0.7469793887704336
[[807 226]
 [130 244]]


In [19]:
model3 = ChurnANN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model3.parameters(), lr=0.001)

In [20]:
epochs = 100

for epoch in range(epochs):
    model3.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        y_pred = model3(X_batch)
        loss = criterion(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Avg Loss: {epoch_loss/len(train_loader):.6f}')

Epoch 10/100 | Avg Loss: 0.421719
Epoch 20/100 | Avg Loss: 0.398895
Epoch 30/100 | Avg Loss: 0.385703
Epoch 40/100 | Avg Loss: 0.378618
Epoch 50/100 | Avg Loss: 0.372664
Epoch 60/100 | Avg Loss: 0.368852
Epoch 70/100 | Avg Loss: 0.365281
Epoch 80/100 | Avg Loss: 0.363029
Epoch 90/100 | Avg Loss: 0.361629
Epoch 100/100 | Avg Loss: 0.358911


In [22]:
model3.eval()
with torch.no_grad():
    y_pred = model3(features_test_tensor)
    y_pred_labels = (y_pred > 0.5).float()

y_pred_np = y_pred_labels.numpy()
y_test_np = target_test_tensor.numpy()

print(accuracy_score(y_test_np, y_pred_np))
print(confusion_matrix(y_test_np, y_pred_np))
print(classification_report(y_test_np, y_pred_np))
print('AUC:', roc_auc_score(y_test_np, y_pred_np))


0.7526652452025586
[[817 216]
 [132 242]]
              precision    recall  f1-score   support

         0.0       0.86      0.79      0.82      1033
         1.0       0.53      0.65      0.58       374

    accuracy                           0.75      1407
   macro avg       0.69      0.72      0.70      1407
weighted avg       0.77      0.75      0.76      1407

AUC: 0.7189795569728376
